# Phân loại bệnh trên lá lúa bằng DenseNet

Notebook này là bản viết lại có thể huấn luyện và dùng mô hình để dự đoán. Mô hình là DenseNet tùy biến theo kiến trúc trong notebook gốc, không phải DenseNet pretrained của torchvision. Các thiết lập chính được giữ lại: ảnh 224 × 224, chia validation 20% với seed 123, batch size 32, Adam với eps=0.05 và 10 epochs.

Trên Kaggle, hãy bật GPU trong Notebook Settings và đính kèm Paddy Disease Classification dataset. Đường dẫn dữ liệu mặc định theo đường dẫn bạn cung cấp. Notebook cũng thử đường dẫn mount mặc định thường gặp trên Kaggle nếu đường dẫn đó không tồn tại. Thư mục train_images phải chứa 10 thư mục lớp ảnh.

In [ ]:
from pathlib import Path
import csv
import json
import time

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt

DATA_ROOT = Path("/kaggle/input/competitions/paddy-disease-classification")
if not DATA_ROOT.exists():
    standard_kaggle_root = Path("/kaggle/input/paddy-disease-classification")
    if standard_kaggle_root.exists():
        DATA_ROOT = standard_kaggle_root

OUTPUT_DIR = (
    Path("/kaggle/working/results")
    if Path("/kaggle/working").exists()
    else Path("results")
)

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3
NUM_CLASSES = 10
BATCH_SIZE = 32
VALIDATION_FRACTION = 0.2
SPLIT_SEED = 123
NUM_WORKERS = 4
EPOCHS = 10
ADAM_EPSILON = 0.05

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Data root: {DATA_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
if DEVICE.type == "cpu":
    print("Đang dùng CPU; huấn luyện DenseNet có thể chậm. Trên Kaggle nên bật GPU.")
# Nếu chạy trên Windows và DataLoader gặp lỗi worker, đặt NUM_WORKERS = 0.

In [ ]:
class OneHotDataset(Dataset):
    """Giữ cách biểu diễn nhãn one-hot của notebook gốc."""

    def __init__(self, dataset, num_classes):
        self.dataset = dataset
        self.num_classes = num_classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, class_index = self.dataset[index]
        target = torch.nn.functional.one_hot(
            torch.tensor(class_index), num_classes=self.num_classes
        ).float()
        return image, target


def resolve_train_images(data_root):
    data_root = Path(data_root)
    train_images = (
        data_root if data_root.name == "train_images" else data_root / "train_images"
    )
    if not train_images.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục ảnh có nhãn: {train_images}. "
            "Hãy kiểm tra dataset đã được đính kèm và DATA_ROOT đúng chưa."
        )
    return train_images


transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
        transforms.ToTensor(),
    ]
)
train_images_dir = resolve_train_images(DATA_ROOT)
full_dataset = datasets.ImageFolder(str(train_images_dir), transform=transform)
class_names = full_dataset.classes
class_to_idx = full_dataset.class_to_idx

if len(class_names) != NUM_CLASSES:
    raise ValueError(
        f"Mong đợi {NUM_CLASSES} lớp, nhưng tìm thấy {len(class_names)}: {class_names}"
    )

validation_size = int(VALIDATION_FRACTION * len(full_dataset))
training_size = len(full_dataset) - validation_size
if validation_size == 0 or training_size == 0:
    raise ValueError("Dataset cần đủ ảnh để tạo cả tập train và validation.")

split_generator = torch.Generator().manual_seed(SPLIT_SEED)
training_subset, validation_subset = random_split(
    full_dataset,
    [training_size, validation_size],
    generator=split_generator,
)
training_dataset = OneHotDataset(training_subset, len(class_names))
validation_dataset = OneHotDataset(validation_subset, len(class_names))

training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Số ảnh: {len(full_dataset)} | train: {training_size} | validation: {validation_size}")
print("Class mapping:", class_to_idx)

In [ ]:
class BottleneckLayer(nn.Module):
    """Tạo growth_rate feature maps mới rồi nối với feature maps đầu vào."""

    def __init__(self, in_channels, growth_rate, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, growth_rate * 4, kernel_size=1, stride=strides, padding=0
        )
        self.bn1 = nn.BatchNorm2d(growth_rate * 4)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(
            growth_rate * 4, growth_rate, kernel_size=3, stride=strides, padding=1
        )
        self.bn2 = nn.BatchNorm2d(growth_rate)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        new_features = self.relu1(self.bn1(self.conv1(x)))
        new_features = self.relu2(self.bn2(self.conv2(new_features)))
        return torch.cat((new_features, x), dim=1)


class TransitionLayer(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU()
        self.conv = nn.Conv2d(
            in_channels, in_channels // 2, kernel_size=1, stride=1, padding=0
        )
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
        self.out_channels = in_channels // 2

    def forward(self, x):
        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        return self.pool(x)


class DenseBlock(nn.Module):
    def __init__(self, in_channels, repetitions, growth_rate=32):
        super().__init__()
        self.layers = nn.ModuleList()
        channels = in_channels
        for _ in range(repetitions):
            self.layers.append(BottleneckLayer(channels, growth_rate))
            channels += growth_rate
        self.out_channels = channels

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class DenseNetB121(nn.Module):
    """DenseNet tùy biến theo notebook gốc, nhận RGB 224x224 và trả logits."""

    def __init__(self, input_shape=(3, 224, 224), num_classes=10, growth_rate=32):
        super().__init__()
        channels = input_shape[0]
        self.bn_init = nn.BatchNorm2d(channels)
        self.relu_init = nn.ReLU()
        self.conv_init = nn.Conv2d(channels, 64, kernel_size=7, stride=2, padding=3)
        self.pool_init = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        channels = 64
        self.dense1 = DenseBlock(channels, repetitions=6, growth_rate=growth_rate)
        channels = self.dense1.out_channels
        self.trans1 = TransitionLayer(channels)
        channels = self.trans1.out_channels

        self.dense2 = DenseBlock(channels, repetitions=12, growth_rate=growth_rate)
        channels = self.dense2.out_channels
        self.trans2 = TransitionLayer(channels)
        channels = self.trans2.out_channels

        self.dense3 = DenseBlock(channels, repetitions=24, growth_rate=growth_rate)
        channels = self.dense3.out_channels
        self.trans3 = TransitionLayer(channels)
        channels = self.trans3.out_channels

        self.dense4 = DenseBlock(channels, repetitions=16, growth_rate=growth_rate)
        channels = self.dense4.out_channels
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.bn_init(x)
        x = self.relu_init(x)
        x = self.conv_init(x)
        x = self.pool_init(x)
        x = self.dense1(x)
        x = self.trans1(x)
        x = self.dense2(x)
        x = self.trans2(x)
        x = self.dense3(x)
        x = self.trans3(x)
        x = self.dense4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [ ]:
def evaluate_model(model, data_loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, one_hot_targets in data_loader:
            images = images.to(device, non_blocking=True)
            one_hot_targets = one_hot_targets.to(device, non_blocking=True)
            logits = model(images)
            loss = loss_function(logits, one_hot_targets)
            batch_size = images.size(0)
            predictions = logits.argmax(dim=1)
            targets = one_hot_targets.argmax(dim=1)
            total_loss += loss.item() * batch_size
            total_correct += (predictions == targets).sum().item()
            total_samples += batch_size

    if total_samples == 0:
        raise ValueError("Validation loader không có batch nào.")
    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }


model = DenseNetB121(
    input_shape=(IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH),
    num_classes=len(class_names),
).to(DEVICE)
optimizer = optim.Adam(model.parameters(), eps=ADAM_EPSILON)
loss_function = nn.CrossEntropyLoss()
history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(f"Model: DenseNetB121 | parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
for epoch in range(EPOCHS):
    started_at = time.time()
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, one_hot_targets in training_loader:
        images = images.to(DEVICE, non_blocking=True)
        one_hot_targets = one_hot_targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_function(logits, one_hot_targets)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        predictions = logits.argmax(dim=1)
        targets = one_hot_targets.argmax(dim=1)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == targets).sum().item()
        total_samples += batch_size

    if total_samples == 0:
        raise ValueError("Training loader không có batch nào.")

    train_loss = total_loss / total_samples
    train_accuracy = total_correct / total_samples
    validation_metrics = evaluate_model(
        model, validation_loader, loss_function, DEVICE
    )
    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["val_loss"].append(validation_metrics["loss"])
    history["val_accuracy"].append(validation_metrics["accuracy"])

    elapsed = time.time() - started_at
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={train_loss:.4f} accuracy={train_accuracy:.4f} | "
        f"val_loss={validation_metrics['loss']:.4f} "
        f"val_accuracy={validation_metrics['accuracy']:.4f} | "
        f"time={elapsed:.1f}s"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / "densenetb121_paddy.pth"
torch.save(model.state_dict(), checkpoint_path)
with (OUTPUT_DIR / "history.json").open("w", encoding="utf-8") as file:
    json.dump(history, file, indent=2)
with (OUTPUT_DIR / "class_to_idx.json").open("w", encoding="utf-8") as file:
    json.dump(class_to_idx, file, indent=2, ensure_ascii=False)

print(f"Đã lưu trọng số: {checkpoint_path}")
print(f"Đã lưu lịch sử và class mapping trong: {OUTPUT_DIR}")

In [ ]:
epochs = range(1, len(history["loss"]) + 1)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history["loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs, history["accuracy"], marker="o", label="train")
axes[1].plot(epochs, history["val_accuracy"], marker="o", label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
figure.tight_layout()
plt.show()

In [ ]:
inference_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
        transforms.ToTensor(),
    ]
)


def load_trained_model(path=checkpoint_path):
    loaded_model = DenseNetB121(
        input_shape=(IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH),
        num_classes=len(class_names),
    ).to(DEVICE)
    try:
        state_dict = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state_dict = torch.load(path, map_location=DEVICE)
    loaded_model.load_state_dict(state_dict)
    loaded_model.eval()
    return loaded_model


def predict_image(image_path, trained_model=None):
    trained_model = trained_model or model
    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB")
    tensor = inference_transform(image).unsqueeze(0).to(DEVICE)
    trained_model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(trained_model(tensor), dim=1)[0]
    predicted_index = int(probabilities.argmax().item())
    return {
        "label": class_names[predicted_index],
        "confidence": float(probabilities[predicted_index].item()),
    }


# Ví dụ: thay đường dẫn bằng một ảnh bạn muốn kiểm tra.
# print(predict_image(train_images_dir / class_names[0] / "ten_anh.jpg"))

In [ ]:
class CompetitionTestDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        with Image.open(self.image_paths[index]) as image_file:
            image = image_file.convert("RGB")
        return inference_transform(image)


def create_submission(data_root=DATA_ROOT, output_path=None, trained_model=None):
    data_root = Path(data_root)
    test_images_dir = data_root / "test_images"
    sample_submission_path = data_root / "sample_submission.csv"
    if not test_images_dir.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục test_images: {test_images_dir}")
    if not sample_submission_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy sample_submission.csv: {sample_submission_path}"
        )

    with sample_submission_path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if not fieldnames or "image_id" not in fieldnames or "label" not in fieldnames:
        raise ValueError("sample_submission.csv cần có cột image_id và label.")

    image_paths = [test_images_dir / Path(row["image_id"]).name for row in rows]
    missing = [path for path in image_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Không tìm thấy ảnh test, ví dụ: {missing[0]}")

    test_loader = DataLoader(
        CompetitionTestDataset(image_paths),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    trained_model = trained_model or model
    trained_model.eval()
    predicted_indices = []
    with torch.no_grad():
        for images in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            predicted_indices.extend(trained_model(images).argmax(dim=1).cpu().tolist())

    for row, class_index in zip(rows, predicted_indices):
        row["label"] = class_names[class_index]

    output_path = Path(output_path) if output_path else OUTPUT_DIR / "submission.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return output_path


sample_submission_path = DATA_ROOT / "sample_submission.csv"
if (DATA_ROOT / "test_images").is_dir() and sample_submission_path.is_file():
    submission_path = create_submission()
    print(f"Đã tạo file submission: {submission_path}")
else:
    print("Không có test_images/sample_submission.csv; bỏ qua bước tạo submission.")